In [1]:
"""
Testing implementation of the FSPro modules
"""

# standard imports
import os
import geopandas as gpd
import pandas as pd

from pathlib import Path
from os.path import join

# Import fb_tools (!!)
from fb_tools import lfps_request, run_flammap_scenarios, list_files

# directories
# use the current working directory
projdir = Path.cwd().parents[1] # moves up two, outside code directory
print(f"Project directory set to: {projdir}")
print(os.getcwd())

# environ vars
proj_crs = 26913  # NAD83 UTM Zone 13N

print("Ready to go !")

Project directory set to: \\Mac\Home\Library\CloudStorage\Box-Box\MCC\fire_modeling\FM_PythonWrapper
\\Mac\Home\Library\CloudStorage\Box-Box\MCC\fire_modeling\FM_PythonWrapper\code\notebooks
Ready to go !


In [2]:
# --- Root of the fb_tools repo (adjust if running from a different location)
REPO_ROOT = Path(projdir)

# Executable
FSPRO_EXE = REPO_ROOT / "code" / "FB" / "bin" / "TestFSPro.exe"

# Sample data
SAMPLE_DIR = REPO_ROOT / "code" / "FB" / "TestFSPro" / "SampleData"
LCP        = SAMPLE_DIR / "416lcp.lcp"
IGN_SHP    = SAMPLE_DIR / "416ign.shp"
INPUT_ORIG = SAMPLE_DIR / "416inputsfile.input"

# Output directory for this test
OUT_DIR = REPO_ROOT / "data" / "fspro_test" / "sample"

# Verify paths exist
for p in [FSPRO_EXE, LCP, IGN_SHP, INPUT_ORIG]:
    status = "OK" if p.exists() else "MISSING"
    print(f"{status}  {p}")

OK  \\Mac\Home\Library\CloudStorage\Box-Box\MCC\fire_modeling\FM_PythonWrapper\code\FB\bin\TestFSPro.exe
OK  \\Mac\Home\Library\CloudStorage\Box-Box\MCC\fire_modeling\FM_PythonWrapper\code\FB\TestFSPro\SampleData\416lcp.lcp
OK  \\Mac\Home\Library\CloudStorage\Box-Box\MCC\fire_modeling\FM_PythonWrapper\code\FB\TestFSPro\SampleData\416ign.shp
OK  \\Mac\Home\Library\CloudStorage\Box-Box\MCC\fire_modeling\FM_PythonWrapper\code\FB\TestFSPro\SampleData\416inputsfile.input


In [3]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Read original, replace IgnitionFile with absolute path
text = INPUT_ORIG.read_text()
text = "\n".join(
    f"IgnitionFile: {IGN_SHP}" if line.strip().startswith("IgnitionFile") else line
    for line in text.splitlines()
)

INPUT_ABS = OUT_DIR / "416inputsfile_abs.input"
INPUT_ABS.write_text(text)
print(f"Input file written to {INPUT_ABS}")

# Confirm the IgnitionFile line
for line in text.splitlines():
    if "ignitionfile" in line.lower():
        print("IgnitionFile line:", line)

Input file written to \\Mac\Home\Library\CloudStorage\Box-Box\MCC\fire_modeling\FM_PythonWrapper\data\fspro_test\sample\416inputsfile_abs.input
IgnitionFile line: IgnitionFile: \\Mac\Home\Library\CloudStorage\Box-Box\MCC\fire_modeling\FM_PythonWrapper\code\FB\TestFSPro\SampleData\416ign.shp


In [4]:
from fb_tools import run_fspro
result = run_fspro(
    fspro_exe=FSPRO_EXE,
    lcp_fp=LCP,
    input_file=INPUT_ABS,
    output_directory=OUT_DIR,
    output_basename="fspro_sample",
)

print(f"Return code: {result.returncode}")

Return code: 0


In [5]:
# Log
log = OUT_DIR / "TestFSPro_run.log"
if log.exists():
    print(log.read_text()[-2000:])  # tail of log
else:
    print("No log file found.")

 fire 47
Launching fire 55...
Completed fire 50
Launching fire 56...
Completed fire 53
Launching fire 57...
Completed fire 49
Launching fire 58...
Completed fire 48
Launching fire 59...
Completed fire 51
Launching fire 60...
Completed fire 54
Launching fire 61...
Completed fire 52
Launching fire 62...
Completed fire 55
Launching fire 63...
Completed fire 56
Launching fire 64...
Completed fire 60
Launching fire 65...
Completed fire 61
Launching fire 66...
Completed fire 57
Launching fire 67...
Completed fire 58
Launching fire 68...
Completed fire 65
Launching fire 69...
Completed fire 63
Launching fire 70...
Completed fire 59
Launching fire 71...
Completed fire 62
Launching fire 72...
Completed fire 64
Launching fire 73...
Completed fire 72
Launching fire 74...
Completed fire 69
Launching fire 75...
Completed fire 68
Launching fire 76...
Completed fire 66
Launching fire 77...
Completed fire 67
Launching fire 78...
Completed fire 71
Launching fire 79...
Completed fire 76
Launching fire 8

In [6]:
# List output files
outputs = sorted(OUT_DIR.glob("fspro_sample*"))
print(f"{len(outputs)} output file(s):")
for f in outputs:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:50s}  {size_kb:8.1f} KB")

26 output file(s):
  fspro_sample_ArrivalDistribution.dbf                  3847.8 KB
  fspro_sample_ArrivalDistribution.prj                     0.6 KB
  fspro_sample_ArrivalDistribution.shp                   419.3 KB
  fspro_sample_ArrivalDistribution.shx                   119.9 KB
  fspro_sample_AvgFlameLength.asc                       7299.7 KB
  fspro_sample_AvgFlameLength.prj                          0.6 KB
  fspro_sample_AvgTime.asc                              7297.7 KB
  fspro_sample_AvgTime.prj                                 0.6 KB
  fspro_sample_BurnProb.asc                             5094.2 KB
  fspro_sample_BurnProb.prj                                0.6 KB
  fspro_sample_ContainSummary.txt                          0.1 KB
  fspro_sample_DailyAcres.txt                             10.1 KB
  fspro_sample_DayTypes.txt                               23.0 KB
  fspro_sample_EventCoverage.txt                          19.9 KB
  fspro_sample_FireStreams.txt                           

In [7]:
import rioxarray as rxr
import matplotlib.pyplot as plt

# FSPro typically outputs a BurnProb or ArrivalTime raster
bp_tif = next(OUT_DIR.glob("*BurnProb*.tif"), None) or next(OUT_DIR.glob("*.tif"), None)

if bp_tif:
    da = rxr.open_rasterio(bp_tif, masked=True).squeeze("band", drop=True)
    fig, ax = plt.subplots(figsize=(7, 6))
    da.plot(ax=ax, cmap="YlOrRd", cbar_kwargs={"label": "Burn Probability"})
    ax.set_title(f"FSPro — {bp_tif.name}")
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()
else:
    print("No GeoTIFF output found — check log for errors or look for ASCII grid outputs.")

ValueError: cannot select a dimension to squeeze out which has length greater than one